In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

kernel = "rbf"
nu_list = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
gamma_list = ["scale", "auto", 0.01, 0.05, 0.1, 0.5, 1.0]

train_path = Path("../data_raw/ECG5000/ECG5000_TRAIN.txt")
test_path = Path("../data_raw/ECG5000/ECG5000_TEST.txt")

results_root = Path("../results/OCSVM_ECG")
results_root.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(train_path, header=None, sep=r"\s+")
test_df = pd.read_csv(test_path, header=None, sep=r"\s+")

df = pd.concat([train_df, test_df], ignore_index=True)

y_raw = df.iloc[:, 0].astype(int)
X = df.iloc[:, 1:].copy()

y_true = (y_raw != 1).astype(int)

row_mean = X.mean(axis=1)
row_std = X.std(axis=1).replace(0, 1e-8)

Xz = X.sub(row_mean, axis=0).div(row_std, axis=0)

X_normal = Xz[y_true == 0].copy()
X_anomaly = Xz[y_true == 1].copy()

X_train = X_normal.copy()

X_test_normal = X_normal.copy()
X_test_anomaly = X_anomaly.sample(n=150, random_state=42)

X_test = pd.concat([X_test_normal, X_test_anomaly], axis=0)

y_test = pd.Series(
    [0] * len(X_test_normal) + [1] * len(X_test_anomaly),
    index=X_test.index
)

test_df_final = X_test.copy()
test_df_final["y_true"] = y_test
test_df_final = test_df_final.sample(frac=1, random_state=42).reset_index(drop=True)

X_test = test_df_final.drop(columns=["y_true"])
y_test = test_df_final["y_true"]

feature_cols = X_train.columns.tolist()

def gamma_to_str(gamma):
    if isinstance(gamma, str):
        return gamma
    return str(gamma).replace(".", "_")

def nu_to_str(nu):
    return str(nu).replace(".", "_")

all_results = []

for gamma in gamma_list:
    gamma_dir = results_root / f"gamma_{gamma_to_str(gamma)}"
    gamma_dir.mkdir(parents=True, exist_ok=True)

    gamma_results = []

    for nu in nu_list:
        exp_name = f"kernel_{kernel}_nu_{nu_to_str(nu)}"
        exp_dir = gamma_dir / exp_name
        exp_dir.mkdir(parents=True, exist_ok=True)

        model = OneClassSVM(
            kernel=kernel,
            nu=nu,
            gamma=gamma
        )

        model.fit(X_train)

        y_pred_raw = model.predict(X_test)
        y_pred = (y_pred_raw == -1).astype(int)

        scores = model.score_samples(X_test)
        decisions = model.decision_function(X_test)

        df_results = X_test.copy()
        df_results["y_true"] = y_test.values
        df_results["ocsvm_pred_raw"] = y_pred_raw
        df_results["ocsvm_score"] = scores
        df_results["ocsvm_decision"] = decisions
        df_results["is_anomaly"] = y_pred

        df_results.to_csv(exp_dir / "predictions.csv", index=False)

        top_anomalies = df_results.sort_values("ocsvm_score").head(20)
        top_anomalies.to_csv(exp_dir / "top20.csv", index=False)

        cm = confusion_matrix(y_test, y_pred)

        accuracy = accuracy_score(y_test, y_pred) * 100
        precision = precision_score(y_test, y_pred, zero_division=0) * 100
        recall = recall_score(y_test, y_pred, zero_division=0) * 100
        f1 = f1_score(y_test, y_pred, zero_division=0) * 100

        result_row = {
            "experiment": exp_name,
            "kernel": kernel,
            "nu": nu,
            "gamma": gamma,
            "accuracy_%": round(accuracy, 2),
            "precision_%": round(precision, 2),
            "recall_%": round(recall, 2),
            "f1_%": round(f1, 2)
        }

        all_results.append(result_row)
        gamma_results.append(result_row)

        metrics_df = pd.DataFrame({
            "Metrika": [
                "Presnosť (Accuracy)",
                "Precíznosť (Precision)",
                "Citlivosť (Recall)",
                "F1-skóre"
            ],
            "Hodnota (%)": [
                round(accuracy, 2),
                round(precision, 2),
                round(recall, 2),
                round(f1, 2)
            ]
        })

        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")

        table = ax.table(
            cellText=metrics_df.values,
            colLabels=metrics_df.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)

        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_text_props(weight="bold")
                cell.set_facecolor("#dce6f1")
            else:
                cell.set_facecolor("#f9f9f9")

        plt.title("Výsledné metriky modelu One-Class SVM – ECG5000", fontsize=12, pad=20)
        plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title("Matica zámen (Confusion Matrix)")
        plt.colorbar()

        classes = ["Normálne", "Anomálie"]
        tick_marks = np.arange(len(classes))

        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)

        threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "black" if cm[i, j] > threshold else "white"
                plt.text(
                    j, i, cm[i, j],
                    ha="center", va="center",
                    color=color
                )

        plt.xlabel("Predikované triedy")
        plt.ylabel("Skutočné triedy")
        plt.tight_layout()
        plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        normal_sample = df_results[df_results["is_anomaly"] == 0][feature_cols].iloc[0].to_numpy() if (df_results["is_anomaly"] == 0).any() else None
        anomaly_sample = df_results[df_results["is_anomaly"] == 1][feature_cols].iloc[0].to_numpy() if (df_results["is_anomaly"] == 1).any() else None

        plt.figure(figsize=(12, 5))
        if normal_sample is not None:
            plt.plot(normal_sample, linewidth=1.5)
        if anomaly_sample is not None:
            plt.plot(anomaly_sample, linewidth=1.5)
        plt.title(f"{exp_name}, gamma={gamma_to_str(gamma)}")
        plt.xlabel("Vzorka signálu")
        plt.ylabel("Normalizovaná hodnota")
        plt.tight_layout()
        plt.savefig(exp_dir / "sample_cycles_plot.png", dpi=300, bbox_inches="tight")
        plt.close()

    gamma_results_df = pd.DataFrame(gamma_results)
    gamma_results_df = gamma_results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
    gamma_results_df.to_csv(gamma_dir / "summary_results.csv", index=False)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Done
